# Q1Lens · 심볼릭 포트 출력

같은 Q1ASM에서 식, 파형 샘플, 물리 포트와 검증 반례를 차례로 확인합니다. 시간 단위는 ns, 진폭 단위는 정규화한 풀스케일입니다.

## 1. 파라미터 없이 실행한 루프

20 ns 재생과 `T` ns 대기를 반복합니다. 마지막 재생은 반복 횟수 × `(20 + T)`에 시작합니다. 32비트 루프 카운터에서 `N=0`은 2³²회를 뜻합니다. 표의 반복 본문은 각 반복 시작을 기준으로 읽습니다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import q1lens
from IPython.display import display

wave = [1.0] * 10 + [-1.0] * 10
memory = {"waveforms": {"bipolar": {"index": 0, "data": wave}}}
probe = {**memory, "program": """move $N,R0
nop
burst: set_awg_gain $G,$G
play 0,0,20
wait $T
loop R0,@burst
set_awg_gain 1000,1000
play 0,0,40
stop"""}
model = q1lens.output(probe)
assert not model.issues
display(model)
start = model.segments[1].start
print("Start =", start.display(), "   Symbols =", sorted(start.free_symbols()))
print("N=0, T=12 →", start.evaluate({"N": 0, "T": 12}), "ns")

## 2. 필요한 시간 창만 그리기

`N=2,000,000`, `T=12`이면 마지막 재생은 64,000,000 ns에 시작합니다. 아래 셀은 그 경계 주변의 샘플 60개만 요청합니다.

In [ ]:
bound = model.bind({"N": 2_000_000, "T": 12, "G": 16384})
edge = 64_000_000
a, b = edge - 24, edge + 36
i, q = bound.samples(a, b, stage="gain_offset")
assert len(i) == 60 and i[edge - a] == 1000 / 32768
def trace(x, series, xlabel="Time (ns)"):
    fig, ax = plt.subplots(figsize=(8, 2.2))
    for label, values in series.items():
        ax.step(x, values, where="post", label=label)
    ax.set(xlabel=xlabel, ylabel="Full scale")
    ax.axhline(0, linewidth=0.5, color="gray")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    plt.show()
trace(range(a-edge, b-edge), {"Path I": i}, "Time relative to last play (ns)")

## 3. 래치와 반복별 gain

프로젝트의 램프는 5회 반복합니다. 첫 샘플에서 읽은 gain은 매회 1/8씩 커집니다. 두 번째 예에서는 `upd_param`이 재생 도중에 gain을 바꾸고 NCO가 I와 Q를 회전시킵니다.

In [ ]:
here = Path.cwd()
folder = next(p for p in (here, here.parent, here / "examples/symbolic-output") if (p / "ramp.q1asm").is_file())
ramp = q1lens.output({**memory, "program": (folder / "ramp.q1asm").read_text()})
i, _ = ramp.samples(0, 100, stage="gain_offset")
print("First samples:", i[::20])
assert i[::20] == [0.125, 0.25, 0.375, 0.5, 0.625]
trace(range(100), {"Ramp · I": i})
modulated = q1lens.output({**memory, "program": """set_awg_gain 8192,0
set_freq 125000000
play 0,0,8
set_awg_gain 16384,0
upd_param 12
stop"""})
base, _ = modulated.samples(0, 20, stage="gain_offset")
rot_i, rot_q = modulated.samples(0, 20, stage="nco")
assert base[7:9] == [0.25, 0.5]
trace(range(20), {"Gain + offset": base, "NCO I": rot_i, "NCO Q": rot_q})

## 4. 같은 포트에 연결한 두 시퀀서

`Mix.out0`에는 `drive.I`와 `boost.Q`가 연결되어 있습니다. 4 ns부터 두 경로의 합이 1.125가 되며 포트 모델에서 1로 클리핑됩니다. 같은 두 경로를 QCoDeS 스냅샷에서 읽으면 YAML과 동일한 포트가 나옵니다.

In [ ]:
from q1timeline.project import load_project_config
from q1timeline.output.modules import ModuleConfig

project = folder / "q1timeline.yml"
config = load_project_config(project)
mix = config.modules["Mix"]
models = {}
for seq in config.sequencers:
    if seq.id in mix.channel_map:
        channel = q1lens.output({**memory, "program": seq.file.read_text()})
        models[seq.id] = channel.bind({k: config.params[k] for k in channel.parameter_names})
before = mix.samples(models, "out0", 0, 24, clip=False)
after = mix.samples(models, "out0", 0, 24)
print("Route:", mix.channel_map, "   At 4 ns:", before[4], "→", after[4])
assert before[4] == 1.125 and after[4] == 1.0
trace(range(24), {"Before clipping": before, "Port output": after})
param = lambda value: {"value": value}
snap = {"parameters": {"out0_offset": param(0.0), "out1_offset": param(0.0)},
        "submodules": {f"sequencer{n}": {"parameters": {"connect_out0": param(path), "connect_out1": param("off")}}
                       for n, path in enumerate(("I", "Q"))}}
restored = ModuleConfig.from_snapshot("Mix", "QRM", snap, {"drive": 0, "boost": 1})
assert restored.samples(models, "out0", 0, 24) == after
print("Snapshot:", restored.channel_map)

## 5. 범위 증명과 Concrete 반례

`drive.q1asm`에 선언한 아홉 개 gain 값에서 피크 상한과 순면적을 검사합니다. `no_clip` 위반이 처음 가능한 시각은 4 ns입니다. VS Code의 CodeLens에서 같은 샘플과 소스 줄을 열 수 있습니다.

In [ ]:
from q1timeline.verification import verify_file

report = verify_file(folder / "drive.q1asm", project=project, include_replay=True)
for check in report["checks"]:
    print(check["status"], "·", check["message"])
assert [c["status"] for c in report["checks"][:3]] == ["proved", "proved", "counterexample"]
w = report["checks"][2]["counterexample"]
print("Concrete:", w["inputs"], "·", w["port"], "·", w["time_ns"], "ns ·", Path(w["file"]).name, "line", w["line"])
assert w["time_ns"] == 4 and w["inputs"] == {"G": 28664}
assert w["timeline"]["output"]["ports"] and abs(w["value"]) > 1

## 6. 컴파일 결과와 Hardware lane

아래 셀은 작은 컴파일 결과 예제에서 프로그램, 업로드한 파형, 연결 설정을 함께 꺼냅니다. 프로젝트를 VS Code에서 열고 **Q1Lens: Open Timeline Preview**를 실행하면 Hardware 뷰의 포트별 파형과 기여 구간을 볼 수 있습니다. 구간 버튼을 누르면 해당 RT 이벤트가 선택됩니다.

In [ ]:
from qbstimeline.compile_worker import extract_q1asm_programs

compiled = {"cluster0": {"slot2": {"instrument_type": "QRM", "sequencers": {
    "seq0": {"sequence": probe, "channel_name": "real_output_0", "connected_output_indices": [0]}
}}}}
program, = extract_q1asm_programs(compiled)
print("/".join(program.path), program.module_type, program.settings)
print("Uploaded samples:", len(program.sequence["waveforms"]["bipolar"]["data"]))
assert q1lens.output(program.sequence).segments == model.segments